In [ ]:
import sys
from pathlib import Path

# Add project root to Python path
sys.path.append(str(Path("..").resolve()))

In [ ]:
from scripts.sra_extractor import SRAExtractor
from Bio import Entrez

In [ ]:
# Initialize extractor
ex = SRAExtractor()

# Prompt for email
email = input("Enter your email (required for NCBI Entrez): ").strip()
Entrez.email = email

# Organism name
organism = input("Enter organism name (e.g., Klebsiella pneumoniae): ").strip()

# Number of samples
while True:
    try:
        retmax = int(input("How many samples to fetch? (default 20): ") or 20)
        break
    except ValueError:
        print("⚠️ Please enter a valid integer")

# Fetch metadata
df = ex.fetch_runinfo(organism, retmax=retmax)

# Save enriched metadata (so we keep size_human, reads, etc.)
fname = f"SraRunInfo_{organism.replace(' ', '_')}.csv"
ex.save_metadata(df, fname)

# Preview all retrieved samples (not just 5)
print(f"\n[INFO] Previewing first {min(retmax, len(df))} metadata records:")
ex.preview_metadata(csvpath=None, n=min(retmax, len(df)))

# Prompt for download
download_prompt = input("\nDo you want to download sequences for these runs? (y/n): ").strip().lower()
if download_prompt in ("y", "yes"):
    run_ids = input("Enter SRR run IDs separated by comma (or leave blank to download all fetched): ").strip()
    if run_ids:
        run_ids = [r.strip() for r in run_ids.split(",")]
    else:
        run_ids = df["Run"].dropna().tolist()
    ex.download_runs(run_ids)
